In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("/workspaces/AnalisisdedatosUQ/Trabajo_infantil1.csv", encoding="latin-1", sep=";")
df.head()


,DIRECTORIO,P6040,P404,P3503,P3271,P400,FEX_C
0,7981549,12,2.0,2,2,3,22.262.628.684
1,7981549,10,2.0,2,2,3,22.262.628.684
2,7981550,11,2.0,2,2,3,26.186.643.482
3,7981550,8,2.0,2,2,3,26.186.643.482
4,7981550,12,2.0,2,2,3,26.186.643.482


1. Verificar que tipos de datos tienen las variables y corregir los que tengan el de dato incorrecto

In [3]:
# Verificar tipos de datos actuales
print('=== TIPOS DE DATOS INICIALES ===')
print(df.dtypes)
print('\n=== MUESTRA DE VALORES ===')
print(df.head())

print('\n=== DETECTANDO Y CORRIGIENDO TIPOS ===')
# Detectar columnas con valores numéricos almacenados como texto
obj_cols = df.select_dtypes(include=['object', 'string']).columns
print(f'Columnas de tipo object/string: {list(obj_cols)}')

for col in obj_cols:
    sample = df[col].dropna().astype(str).head(20)
    if sample.empty:
        print(f'{col}: columna vacía, saltando')
        continue
    
    # Limpiar separadores de miles y comas decimales
    cleaned = sample.str.replace(r'\.', '', regex=True).str.replace(',', '.', regex=False)
    
    # Verificar si todos los valores son numéricos
    is_numeric = cleaned.str.match(r'^[-+]?[0-9]+(\.[0-9]+)?$').all()
    
    if is_numeric:
        print(f'{col}: convirtiendo a numérico')
        df[col] = pd.to_numeric(df[col].astype(str).str.replace(r'\.', '', regex=True).str.replace(',', '.', regex=False), errors='coerce')
    else:
        print(f'{col}: manteniendo como texto')

# Asegurar que DIRECTORIO quede como cadena si es identificador
if 'DIRECTORIO' in df.columns:
    df['DIRECTORIO'] = df['DIRECTORIO'].astype(str)
    print('DIRECTORIO: convertido a string (identificador)')

print('\n=== TIPOS DE DATOS CORREGIDOS ===')
print(df.dtypes)

print('\n=== VERIFICACIÓN FINAL ===')
print('Valores nulos por columna:')
print(df.isnull().sum())

=== TIPOS DE DATOS INICIALES ===
DIRECTORIO      int64
P6040           int64
P404          float64
P3503           int64
P3271           int64
P400            int64
FEX_C             str
dtype: object

=== MUESTRA DE VALORES ===
   DIRECTORIO  P6040  P404  P3503  P3271  P400           FEX_C
0     7981549     12   2.0      2      2     3  22.262.628.684
1     7981549     10   2.0      2      2     3  22.262.628.684
2     7981550     11   2.0      2      2     3  26.186.643.482
3     7981550      8   2.0      2      2     3  26.186.643.482
4     7981550     12   2.0      2      2     3  26.186.643.482

=== DETECTANDO Y CORRIGIENDO TIPOS ===
Columnas de tipo object/string: ['FEX_C']
FEX_C: convirtiendo a numérico
DIRECTORIO: convertido a string (identificador)

=== TIPOS DE DATOS CORREGIDOS ===
DIRECTORIO        str
P6040           int64
P404          float64
P3503           int64
P3271           int64
P400            int64
FEX_C           int64
dtype: object

=== VERIFICACIÓN FINAL ===
V

2. Particionar el dataset en entrenamiento y prueba

In [9]:
# Instalar sklearn si no está disponible
import subprocess
import sys

try:
    from sklearn.model_selection import train_test_split
    print('✅ Scikit-learn ya está instalado')
except ImportError:
    print('Instalando scikit-learn...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'scikit-learn'])
    from sklearn.model_selection import train_test_split
    print('✅ Scikit-learn instalado correctamente')

Instalando scikit-learn...
  Obtaining dependency information for scikit-learn from https://files.pythonhosted.org/packages/97/74/b7a304feb2b49df9fafa9382d4d09061a96ee9a9449a7cbea7988dda0828/scikit_learn-1.8.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata
  Obtaining dependency information for scipy>=1.10.0 from https://files.pythonhosted.org/packages/01/8e/1e35281b8ab6d5d72ebe9911edcdffa3f36b04ed9d51dec6dd140396e220/scipy-1.17.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 3.8 MB/s eta 0:00:00
  Obtaining dependency information for joblib>=1.3.0 from https://files.pythonhosted.org/packages/7b/91/984aca2ec129e2757d1e4e3c81c3fcda9d0f85b74670a094cc443d9ee949/joblib-1.5.3-py3-none-any.whl.metadata
  Obtaining dependency information for threadpoolctl>=3.2.0 from https://files.pythonhosted.org/packages/32/d5/f9a850d79b0851d1d4ef6456097579a9005b31fea68726a4ae5f2d82ddd9/threadpoolctl-3.6.


[notice] A new release of pip is available: 23.2.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


✅ Scikit-learn instalado correctamente


In [10]:
# Particionar el dataset en entrenamiento y prueba
from sklearn.model_selection import train_test_split

# Verificar que df existe
if 'df' not in locals():
    print('❌ Error: df no está definido. Ejecuta las celdas anteriores primero.')
else:
    # Particionar el dataset
    train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
    
    print('=' * 50)
    print('PARTICIÓN DE DATOS COMPLETADA')
    print('=' * 50)
    
    print(f'📊 Dataset original: {df.shape[0]:,} filas × {df.shape[1]} columnas')
    print()
    
    print(f'🎯 Entrenamiento: {train_df.shape[0]:,} filas ({len(train_df)/len(df)*100:.1f}%)')
    print(f'🧪 Prueba: {test_df.shape[0]:,} filas ({len(test_df)/len(df)*100:.1f}%)')
    print()
    
    # Mostrar distribución de la variable objetivo (edad)
    if 'P6040' in df.columns:
        print('📈 DISTRIBUCIÓN DE EDADES (P6040):')
        print('-' * 40)
        
        # Crear tabla comparativa
        train_dist = train_df['P6040'].value_counts().sort_index()
        test_dist = test_df['P6040'].value_counts().sort_index()
        
        print(f'{"Edad":<5} {"Entrenamiento":<12} {"Prueba":<8} {"Total":<8}')
        print('-' * 40)
        
        for edad in sorted(set(train_dist.index) | set(test_dist.index)):
            train_count = train_dist.get(edad, 0)
            test_count = test_dist.get(edad, 0)
            total = train_count + test_count
            print(f'{edad:<5} {train_count:<12,} {test_count:<8,} {total:<8,}')
        
        print()
        print('✅ Partición exitosa - Listo para modelado!')

PARTICIÓN DE DATOS COMPLETADA
📊 Dataset original: 38,956 filas × 7 columnas

🎯 Entrenamiento: 31,164 filas (80.0%)
🧪 Prueba: 7,792 filas (20.0%)

📈 DISTRIBUCIÓN DE EDADES (P6040):
----------------------------------------
Edad  Entrenamiento Prueba   Total   
----------------------------------------
5     2,043        527      2,570   
6     2,200        536      2,736   
7     2,317        555      2,872   
8     2,339        559      2,898   
9     2,349        636      2,985   
10    2,445        605      3,050   
11    2,342        567      2,909   
12    2,516        619      3,135   
13    2,431        608      3,039   
14    2,420        612      3,032   
15    2,475        659      3,134   
16    2,673        671      3,344   
17    2,614        638      3,252   

✅ Partición exitosa - Listo para modelado!


3. Guardar el conjunto de entrenamiento y prueba en archivos separados (train.csv, test.csv)

In [11]:
import os
from pathlib import Path

if 'train_df' in locals() and 'test_df' in locals():
    # Definir rutas de guardado
    output_dir = Path("/workspaces/AnalisisdedatosUQ/datos_procesados")
    output_dir.mkdir(exist_ok=True)
    
    train_path = output_dir / "train.csv"
    test_path = output_dir / "test.csv"
    
    # Guardar los conjuntos
    train_df.to_csv(train_path, index=False, sep=';', encoding='utf-8')
    test_df.to_csv(test_path, index=False, sep=';', encoding='utf-8')
    
    print('✅ Archivos guardados exitosamente!')
    print()
    print(f'📁 Entrenamiento: {train_path}')
    print(f'   Tamaño: {train_path.stat().st_size / 1024:.2f} KB ({train_df.shape[0]} filas)')
    print()
    print(f'📁 Prueba: {test_path}')
    print(f'   Tamaño: {test_path.stat().st_size / 1024:.2f} KB ({test_df.shape[0]} filas)')
    print()
    print('✅ Datos listos para análisis y modelado!')
else:
    print('❌ Error: train_df o test_df no están definidos.')
    print('Ejecuta la celda de partición primero.')

✅ Archivos guardados exitosamente!

📁 Entrenamiento: /workspaces/AnalisisdedatosUQ/datos_procesados/train.csv
   Tamaño: 987.68 KB (31164 filas)

📁 Prueba: /workspaces/AnalisisdedatosUQ/datos_procesados/test.csv
   Tamaño: 247.05 KB (7792 filas)

✅ Datos listos para análisis y modelado!


4. Aplicar One-Hot Encoding a variables categóricas

In [12]:
from sklearn.preprocessing import OneHotEncoder
import pandas as pd

if 'train_df' in locals() and 'test_df' in locals():
    print('=' * 60)
    print('ONE-HOT ENCODING DE VARIABLES CATEGÓRICAS')
    print('=' * 60)
    
    # Copiar datos para no modificar los originales
    train_encoded = train_df.copy()
    test_encoded = test_df.copy()
    
    # Identificar variables categóricas (excluyendo identificadores)
    exclude_cols = ['DIRECTORIO', 'FEX_C']
    cat_cols = [col for col in train_df.columns 
                if train_df[col].dtype in ['int64', 'float64'] 
                and col not in exclude_cols
                and train_df[col].nunique() < 50]  # Menos de 50 valores únicos
    
    print(f'\n📊 Variables categóricas encontradas:')
    for col in cat_cols:
        n_unique = train_df[col].nunique()
        print(f'   • {col}: {n_unique} valores únicos {sorted(train_df[col].dropna().unique())}')
    
    # Aplicar OneHotEncoder
    if cat_cols:
        encoder = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')
        
        # Ajustar el encoder con datos de entrenamiento
        train_encoded_arr = encoder.fit_transform(train_df[cat_cols])
        test_encoded_arr = encoder.transform(test_df[cat_cols])
        
        # Obtener nombres de las nuevas columnas
        feature_names = encoder.get_feature_names_out(cat_cols)
        
        # Crear dataframes con las nuevas columnas
        train_encoded_df = pd.DataFrame(train_encoded_arr, columns=feature_names, index=train_df.index)
        test_encoded_df = pd.DataFrame(test_encoded_arr, columns=feature_names, index=test_df.index)
        
        # Mantener columnas numéricas que no fueron codificadas
        numeric_cols = [col for col in train_df.columns if col not in cat_cols]
        
        train_final = pd.concat([train_df[numeric_cols], train_encoded_df], axis=1)
        test_final = pd.concat([test_df[numeric_cols], test_encoded_df], axis=1)
        
        print(f'\n✅ One-Hot Encoding aplicado exitosamente!')
        print(f'\nDimensiones después del encoding:')
        print(f'   Entrenamiento: {train_final.shape[0]} filas × {train_final.shape[1]} columnas')
        print(f'   Prueba: {test_final.shape[0]} filas × {test_final.shape[1]} columnas')
        
        print(f'\n🆕 Primeras columnas después del encoding:')
        print(train_final.head(3).to_string())
        
        # Guardar versiones codificadas
        from pathlib import Path
        output_dir = Path("/workspaces/AnalisisdedatosUQ/datos_procesados")
        output_dir.mkdir(exist_ok=True)
        
        train_final.to_csv(output_dir / "train_encoded.csv", index=False, sep=';')
        test_final.to_csv(output_dir / "test_encoded.csv", index=False, sep=';')
        
        print(f'\n💾 Archivos codificados guardados:')
        print(f'   • datos_procesados/train_encoded.csv')
        print(f'   • datos_procesados/test_encoded.csv')
        
    else:
        print('\n⚠️ No hay variables categóricas para codificar.')
        
else:
    print('❌ Error: train_df o test_df no están definidos.')
    print('Ejecuta las celdas anteriores primero.')

ONE-HOT ENCODING DE VARIABLES CATEGÓRICAS

📊 Variables categóricas encontradas:
   • P6040: 13 valores únicos [np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17)]
   • P404: 2 valores únicos [np.float64(1.0), np.float64(2.0)]
   • P3503: 2 valores únicos [np.int64(1), np.int64(2)]
   • P3271: 2 valores únicos [np.int64(1), np.int64(2)]
   • P400: 6 valores únicos [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]

✅ One-Hot Encoding aplicado exitosamente!

Dimensiones después del encoding:
   Entrenamiento: 31164 filas × 23 columnas
   Prueba: 7792 filas × 23 columnas

🆕 Primeras columnas después del encoding:
      DIRECTORIO        FEX_C  P6040_6  P6040_7  P6040_8  P6040_9  P6040_10  P6040_11  P6040_12  P6040_13  P6040_14  P6040_15  P6040_16  P6040_17  P404_2.0  P404_nan  P3503_2  P3271_2  P400_2  P400_3  P400_4  P400_5  P400_6
20230

5. Para las variables numericas aplicar escalonamiento o normalizar

In [13]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import pandas as pd
import numpy as np

if 'train_final' in locals() and 'test_final' in locals():
    print('=' * 60)
    print('NORMALIZACIÓN DE VARIABLES NUMÉRICAS')
    print('=' * 60)
    
    train_normalized = train_final.copy()
    test_normalized = test_final.copy()
    
    # Identificar columnas numéricas excluyendo identificadores
    exclude_cols = ['DIRECTORIO', 'FEX_C']
    numeric_cols = [col for col in train_final.columns 
                    if train_final[col].dtype in ['int64', 'float64'] 
                    and col not in exclude_cols]
    
    print(f'\n📊 Variables numéricas a normalizar:')
    for col in numeric_cols:
        mean = train_final[col].mean()
        std = train_final[col].std()
        min_val = train_final[col].min()
        max_val = train_final[col].max()
        print(f'   • {col}: media={mean:.2f}, std={std:.2f}, rango=[{min_val:.0f}, {max_val:.0f}]')
    
    # Aplicar StandardScaler (media=0, desv.est=1)
    if numeric_cols:
        scaler = StandardScaler()
        
        # Ajustar el scaler con datos de entrenamiento
        train_scaled_arr = scaler.fit_transform(train_final[numeric_cols])
        test_scaled_arr = scaler.transform(test_final[numeric_cols])
        
        # Crear dataframes con variables normalizadas
        train_scaled_df = pd.DataFrame(train_scaled_arr, columns=numeric_cols, index=train_final.index)
        test_scaled_df = pd.DataFrame(test_scaled_arr, columns=numeric_cols, index=test_final.index)
        
        # Mantener columnas que no fueron normalizadas (identificadores)
        other_cols = [col for col in train_final.columns if col not in numeric_cols]
        
        train_final_normalized = pd.concat([train_final[other_cols], train_scaled_df], axis=1)
        test_final_normalized = pd.concat([test_final[other_cols], test_scaled_df], axis=1)
        
        print(f'\n✅ Normalización aplicada exitosamente!')
        print(f'   Método: StandardScaler (media=0, desviación estándar=1)')
        
        print(f'\n📈 Variables después de la normalización (primeras 5 filas):')
        print(train_final_normalized[numeric_cols].head().to_string())
        
        print(f'\n✔️ Nuevas estadísticas (después de normalizar):')
        print(train_final_normalized[numeric_cols].describe().to_string())
        
        # Guardar versiones normalizadas
        from pathlib import Path
        output_dir = Path("/workspaces/AnalisisdedatosUQ/datos_procesados")
        output_dir.mkdir(exist_ok=True)
        
        train_final_normalized.to_csv(output_dir / "train_normalized.csv", index=False, sep=';')
        test_final_normalized.to_csv(output_dir / "test_normalized.csv", index=False, sep=';')
        
        print(f'\n💾 Archivos normalizados guardados:')
        print(f'   • datos_procesados/train_normalized.csv ({train_final_normalized.shape})')
        print(f'   • datos_procesados/test_normalized.csv ({test_final_normalized.shape})')
        
        print(f'\n✅ Datos listos para modelado!')
        
    else:
        print('\n⚠️ No hay variables numéricas para normalizar.')
        
elif 'train_df' in locals() and 'test_df' in locals():
    print('⚠️ Nota: Ejecuta primero la celda de One-Hot Encoding')
    print('para aplicar normalización sobre datos codificados.')
else:
    print('❌ Error: train_final o test_final no están definidos.')
    print('Ejecuta las celdas anteriores en orden.')

NORMALIZACIÓN DE VARIABLES NUMÉRICAS

📊 Variables numéricas a normalizar:
   • P6040_6: media=0.07, std=0.26, rango=[0, 1]
   • P6040_7: media=0.07, std=0.26, rango=[0, 1]
   • P6040_8: media=0.08, std=0.26, rango=[0, 1]
   • P6040_9: media=0.08, std=0.26, rango=[0, 1]
   • P6040_10: media=0.08, std=0.27, rango=[0, 1]
   • P6040_11: media=0.08, std=0.26, rango=[0, 1]
   • P6040_12: media=0.08, std=0.27, rango=[0, 1]
   • P6040_13: media=0.08, std=0.27, rango=[0, 1]
   • P6040_14: media=0.08, std=0.27, rango=[0, 1]
   • P6040_15: media=0.08, std=0.27, rango=[0, 1]
   • P6040_16: media=0.09, std=0.28, rango=[0, 1]
   • P6040_17: media=0.08, std=0.28, rango=[0, 1]
   • P404_2.0: media=0.97, std=0.17, rango=[0, 1]
   • P404_nan: media=0.02, std=0.15, rango=[0, 1]
   • P3503_2: media=1.00, std=0.03, rango=[0, 1]
   • P3271_2: media=0.49, std=0.50, rango=[0, 1]
   • P400_2: media=0.00, std=0.03, rango=[0, 1]
   • P400_3: media=0.86, std=0.35, rango=[0, 1]
   • P400_4: media=0.04, std=0.19, r